# 問題
61で構築した学習データの特徴ベクトルを用いて、ロジスティック回帰モデルを学習せよ。

In [1]:
import spacy
from collections import Counter

# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"


# --- 形態素解析の準備 ---
nlp = spacy.load("en_core_web_sm")

def integrate_dict(path):
  with open(path, 'r', encoding='utf-8') as f:
    final_list = []
    for row in f:
      word_counter = Counter()
      row = row.strip().split("\t")
      tmp_dict = {}
      if not row:
          continue
      # カテゴリ宣言行
      if row[1] == "0" or row[1] == "1":
        text = row[0]
        # 形態素解析を行い、単語をカウント
        doc = nlp(text)
        for tok in doc:
          # 追加: 軽い前処理（英字のみ＋小文字化）
          if tok.is_alpha:
            word_counter.update([tok.text.lower()])
            tmp_dict["text"] = row[0]
            tmp_dict["label"] = row[1]
            tmp_dict["feature"] = dict(word_counter)
            final_list.append(tmp_dict)
      else:
        print("0と1以外です：", row[1])
    return final_list

In [2]:
integrate_dict(path_train)

0と1以外です： label


[{'text': 'hide new secretions from the parental units ',
  'label': '0',
  'feature': {'hide': 1,
   'new': 1,
   'secretions': 1,
   'from': 1,
   'the': 1,
   'parental': 1,
   'units': 1}},
 {'text': 'hide new secretions from the parental units ',
  'label': '0',
  'feature': {'hide': 1,
   'new': 1,
   'secretions': 1,
   'from': 1,
   'the': 1,
   'parental': 1,
   'units': 1}},
 {'text': 'hide new secretions from the parental units ',
  'label': '0',
  'feature': {'hide': 1,
   'new': 1,
   'secretions': 1,
   'from': 1,
   'the': 1,
   'parental': 1,
   'units': 1}},
 {'text': 'hide new secretions from the parental units ',
  'label': '0',
  'feature': {'hide': 1,
   'new': 1,
   'secretions': 1,
   'from': 1,
   'the': 1,
   'parental': 1,
   'units': 1}},
 {'text': 'hide new secretions from the parental units ',
  'label': '0',
  'feature': {'hide': 1,
   'new': 1,
   'secretions': 1,
   'from': 1,
   'the': 1,
   'parental': 1,
   'units': 1}},
 {'text': 'hide new secretions

In [8]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# データ作成
train_items = integrate_dict(path_train)
dev_items   = integrate_dict(path_dev)

X_train_dicts = [d["feature"] for d in train_items]
y_train = np.array([int(d["label"]) for d in train_items])

X_dev_dicts = [d["feature"] for d in dev_items]
y_dev = np.array([int(d["label"]) for d in dev_items])

# 辞書 → 疎行列
vec = DictVectorizer(sparse=True)
X_train = vec.fit_transform(X_train_dicts)
X_dev   = vec.transform(X_dev_dicts)

# 学習（ロジスティック回帰）
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# 評価
pred = clf.predict(X_dev)
print("Accuracy:", accuracy_score(y_dev, pred))
print(classification_report(y_dev, pred, digits=3))

0と1以外です： label
0と1以外です： label
Accuracy: 0.7968614357262104
              precision    recall  f1-score   support

           0      0.818     0.746     0.780      7241
           1      0.780     0.845     0.811      7734

    accuracy                          0.797     14975
   macro avg      0.799     0.795     0.796     14975
weighted avg      0.798     0.797     0.796     14975

